# Name That LLM — V31 plain LoRA member

Independent port of the plain `lora24_claude` member from
[V31](https://www.kaggle.com/kernels/scriptcontent/372363760/download).
Attach the competition, Qwen3-8B Base v1 and Zbigniew Kirchner's Apache 2.0
LoRA dataset v1. Use a GPU with Internet disabled. The Notebook scores the
mounted test rows directly and does not fit on competition training data.


In [ ]:
import hashlib
import json
import os
import sys
import tempfile
from pathlib import Path

os.environ.update(HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1')
if 'ntl_repro' in sys.modules or 'ntl_baseline' in sys.modules:
    raise RuntimeError('Restart the kernel before running this Notebook.')
SOURCES = {'src/ntl_baseline/__init__.py': '"""Official-baseline reproduction; no data access at import time."""\n\n__version__ = "0.1.0"\n', 'src/ntl_baseline/constants.py': '"""Exact prediction labels from the competition\'s Evaluation page."""\n\nLABELS: tuple[str, ...] = tuple(\n    sorted(\n        [\n            "gpt-5.5",\n            "gpt-5.6-luna",\n            "gpt-5.6-sol",\n            "gpt-5.6-terra",\n            "gpt-6-astra",\n            "gemini-3.1-pro-preview",\n            "gemini-3.6-flash",\n            "gemini-3.7-flash",\n            "gemini-3.8-flash",\n            "claude-fable-5",\n            "claude-opus-4-6",\n            "claude-opus-4-8",\n            "claude-opus-5",\n            "claude-sonnet-5",\n            "qwen3.8-max",\n            "glm-5.3",\n            "glm-5.3-flash",\n            "kimi-k3",\n        ]\n    )\n)\n\nDATA_FILES = ("train.csv", "test.csv", "sample_submission.parquet")\n\n\nclass ContractError(ValueError):\n    """Invalid data, configuration, artifact or submission contract."""\n', 'src/ntl_baseline/data.py': '"""Local-only input, archive and submission contracts. Never log passage content."""\n\nimport csv\nimport hashlib\nimport json\nimport stat\nimport tempfile\nimport zipfile\nfrom datetime import UTC, datetime\nfrom pathlib import Path, PurePosixPath\n\nimport pandas as pd\n\nfrom .constants import DATA_FILES, LABELS, ContractError\n\nMAX_INPUT_BYTES = 64 * 1024 * 1024\n\n\ndef file_sha256(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef _check_columns(frame: pd.DataFrame, expected: tuple[str, ...]) -> None:\n    if not frame.columns.is_unique or set(frame.columns) != set(expected):\n        raise ContractError(f"columns must be exactly {\', \'.join(expected)}")\n    if frame.empty:\n        raise ContractError("input must contain at least one row")\n\n\ndef _check_strings(series: pd.Series, column: str) -> None:\n    if (\n        series.isna().any()\n        or not series.map(lambda value: isinstance(value, str) and bool(value.strip())).all()\n    ):\n        raise ContractError(f"{column} must contain non-empty strings")\n\n\ndef _check_ids(frame: pd.DataFrame) -> None:\n    _check_strings(frame["id"], "id")\n    if frame["id"].duplicated().any():\n        raise ContractError("id must be unique")\n\n\ndef _normalize_text(frame: pd.DataFrame) -> pd.DataFrame:\n    result = frame.copy(deep=True).reset_index(drop=True)\n    result["text"] = result["text"].fillna("")\n    if not result["text"].map(lambda value: isinstance(value, str)).all():\n        raise ContractError("text must contain strings or missing values")\n    return result\n\n\ndef validate_train(frame: pd.DataFrame) -> pd.DataFrame:\n    _check_columns(frame, ("id", "prompt_id", "text", "model"))\n    _check_ids(frame)\n    _check_strings(frame["prompt_id"], "prompt_id")\n    _check_strings(frame["model"], "model label")\n    if not frame["model"].isin(LABELS).all():\n        raise ContractError("unknown model label")\n    if set(frame["model"]) != set(LABELS):\n        raise ContractError("training data must cover all 18 labels")\n    return _normalize_text(frame)\n\n\ndef validate_test(frame: pd.DataFrame) -> pd.DataFrame:\n    _check_columns(frame, ("id", "text"))\n    _check_ids(frame)\n    return _normalize_text(frame)\n\n\ndef validate_submission(frame: pd.DataFrame, test: pd.DataFrame) -> pd.DataFrame:\n    test = validate_test(test)\n    _check_columns(frame, ("id", "model"))\n    _check_ids(frame)\n    _check_strings(frame["model"], "model label")\n    if not frame["model"].isin(LABELS).all():\n        raise ContractError("submission contains an invalid model label")\n    if len(frame) != len(test) or set(frame["id"]) != set(test["id"]):\n        raise ContractError("submission id set must equal test id set")\n    return frame.set_index("id").loc[test["id"], ["model"]].reset_index()\n\n\ndef input_file(data_dir: Path, name: str) -> Path:\n    path = data_dir / name\n    if path.is_symlink() or not path.is_file():\n        raise ContractError(f"missing regular input file: {name}")\n    if path.stat().st_size > MAX_INPUT_BYTES:\n        raise ContractError(f"input exceeds local size budget: {name}")\n    return path\n\n\ndef read_csv_input(data_dir: Path, name: str) -> pd.DataFrame:\n    path = input_file(data_dir, name)\n    try:\n        with path.open(encoding="utf-8", newline="") as stream:\n            header = next(csv.reader(stream), [])\n        if not header or len(set(header)) != len(header):\n            raise ContractError(f"invalid or duplicate columns in {name}")\n        # Enforce the published string schema (including leading-zero IDs).\n        # Missing text is still normalized with the official fillna(\'\') policy.\n        return pd.read_csv(path, dtype={name: str for name in ("id", "prompt_id", "text", "model")})\n    except (pd.errors.ParserError, pd.errors.EmptyDataError, UnicodeError) as exc:\n        raise ContractError(f"invalid CSV input: {name}") from exc\n\n\ndef load_test(data_dir: Path) -> pd.DataFrame:\n    return validate_test(read_csv_input(data_dir, "test.csv"))\n\n\ndef load_competition_data(\n    data_dir: Path, *, require_sample: bool = True\n) -> tuple[pd.DataFrame, pd.DataFrame]:\n    train = validate_train(read_csv_input(data_dir, "train.csv"))\n    test = load_test(data_dir)\n    if not require_sample:\n        # Hidden reruns replace test.csv; the public sample is not its ID authority.\n        return train, test\n    sample_path = input_file(data_dir, "sample_submission.parquet")\n    try:\n        sample = pd.read_parquet(sample_path, engine="pyarrow")\n    except (ValueError, OSError) as exc:\n        raise ContractError("invalid sample_submission.parquet") from exc\n    validate_submission(sample, test)\n    return train, test\n\n\ndef import_archive(archive: Path, destination: Path) -> dict[str, object]:\n    if destination.exists() and (not destination.is_dir() or any(destination.iterdir())):\n        raise FileExistsError(f"refusing to replace existing data: {destination}")\n    if destination.is_symlink() or not archive.is_file():\n        raise ContractError("archive and destination must be local regular paths")\n    if archive.stat().st_size > MAX_INPUT_BYTES:\n        raise ContractError("archive exceeds local size budget")\n    try:\n        with zipfile.ZipFile(archive) as bundle:\n            members = bundle.infolist()\n            names = [member.filename for member in members]\n            for member in members:\n                path = PurePosixPath(member.filename)\n                is_link = stat.S_ISLNK(member.external_attr >> 16)\n                if path.is_absolute() or ".." in path.parts or "\\\\" in member.filename or is_link:\n                    raise ContractError("unsafe archive member path or symlink")\n            if len(members) != 3 or set(names) != set(DATA_FILES):\n                raise ContractError("archive must contain exactly the three official files at root")\n            if sum(member.file_size for member in members) > MAX_INPUT_BYTES:\n                raise ContractError("archive expansion exceeds local size budget")\n            destination.parent.mkdir(parents=True, exist_ok=True)\n            with tempfile.TemporaryDirectory(prefix=".ntl-import-", dir=destination.parent) as temp:\n                staging = Path(temp) / "data"\n                staging.mkdir()\n                for member in members:\n                    payload = bundle.read(member)\n                    if len(payload) != member.file_size:\n                        raise ContractError("archive member size mismatch")\n                    (staging / member.filename).write_bytes(payload)\n                load_competition_data(staging)\n                manifest: dict[str, object] = {\n                    "source_url": "https://www.kaggle.com/competitions/name-that-llm/data",\n                    "imported_at_utc": datetime.now(UTC).isoformat(),\n                    "archive_name": archive.name,\n                    "archive_sha256": file_sha256(archive),\n                    "files": {name: file_sha256(staging / name) for name in DATA_FILES},\n                }\n                (staging / "source.json").write_text(\n                    json.dumps(manifest, indent=2) + "\\n", encoding="utf-8"\n                )\n                staging.rename(destination)\n                return manifest\n    except zipfile.BadZipFile as exc:\n        raise ContractError("invalid ZIP archive") from exc\n', 'src/ntl_repro/__init__.py': '"""Offline single-GPU inference for a V31 classifier member."""\n', 'src/ntl_repro/branch.py': '"""Numerical and tabular contracts shared by server and notebook callers."""\n\nfrom collections.abc import Callable\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport numpy.typing as npt\nimport pandas as pd\n\nfrom ntl_baseline.constants import LABELS, ContractError\nfrom ntl_baseline.data import validate_submission, validate_test\n\nPOOL_LAYERS = (2, 4, 6, 8, 12, 16, 20, 24)\nHIDDEN_SIZE = 4096\nFEATURE_SIZE = HIDDEN_SIZE * len(POOL_LAYERS)\nTOKEN_LIMIT = 447\nCLAUDE_LABELS = tuple(label for label in LABELS if label.startswith("claude-"))\nArray = npt.NDArray[np.floating[Any]]\n\n\ndef _require_finite(values: Array, shape: tuple[int, ...], name: str) -> None:\n    if values.shape != shape or values.dtype.kind != "f" or not np.isfinite(values).all():\n        raise ContractError(f"{name} must be finite floating-point values with shape {shape}")\n\n\n@dataclass(frozen=True)\nclass PlainReadout:\n    coefficients: Array\n    intercept: Array\n\n    def logits(self, features: Array) -> Array:\n        _require_finite(features, (len(features), FEATURE_SIZE), "pooled features")\n        return features @ self.coefficients.T + self.intercept\n\n\ndef load_readout(path: Path) -> PlainReadout:\n    with np.load(path, allow_pickle=False) as archive:\n        if tuple(archive["classes"]) != LABELS:\n            raise ContractError("readout class order differs from V31")\n        if tuple(archive["pool_layers"]) != POOL_LAYERS:\n            raise ContractError("readout pooling layers differ from V31")\n        coefficients, intercept = archive["plain_coef"], archive["plain_intercept"]\n    _require_finite(coefficients, (len(LABELS), FEATURE_SIZE), "plain coefficients")\n    _require_finite(intercept, (len(LABELS),), "plain intercept")\n    return PlainReadout(coefficients, intercept)\n\n\n@dataclass(frozen=True)\nclass AuxiliaryReadouts:\n    center: PlainReadout\n    claude: PlainReadout\n\n\ndef load_auxiliary_readouts(path: Path) -> AuxiliaryReadouts:\n    with np.load(path, allow_pickle=False) as archive:\n        if tuple(archive["classes"]) != LABELS or tuple(archive["claude_classes"]) != CLAUDE_LABELS:\n            raise ContractError("auxiliary readout class order differs from V31")\n        if tuple(archive["pool_layers"]) != POOL_LAYERS:\n            raise ContractError("auxiliary readout pooling layers differ from V31")\n        center = PlainReadout(archive["center_coef"], archive["center_intercept"])\n        claude = PlainReadout(archive["claude_coef"], archive["claude_intercept"])\n    for name, head, count in (\n        ("center", center, len(LABELS)),\n        ("claude", claude, len(CLAUDE_LABELS)),\n    ):\n        _require_finite(head.coefficients, (count, FEATURE_SIZE), f"{name} coefficients")\n        _require_finite(head.intercept, (count,), f"{name} intercept")\n    return AuxiliaryReadouts(center, claude)\n\n\ndef auxiliary_outputs(\n    ids: list[str], features: Array, readouts: AuxiliaryReadouts, groups: list[str] | None\n) -> tuple[pd.DataFrame, pd.DataFrame]:\n    if len(ids) != len(features) or len(set(ids)) != len(ids):\n        raise ContractError("auxiliary scores need one unique ID per feature row")\n    claude = _score_frame(ids, _log_softmax(readouts.claude.logits(features)), CLAUDE_LABELS)\n    if groups is None:\n        return claude, _score_frame([], np.empty((0, len(LABELS))), LABELS)\n    if len(groups) != len(ids):\n        raise ContractError("expected one group per feature row")\n    group_rows: dict[str, list[int]] = {}\n    for row, group in enumerate(groups):\n        group_rows.setdefault(group, []).append(row)\n    eligible = [rows for rows in group_rows.values() if 2 <= len(rows) <= 18]\n    centered = features.astype(np.float32, copy=True)\n    for rows in eligible:\n        centered[rows] -= centered[rows].mean(axis=0)\n    rows = sorted(row for group in eligible for row in group)\n    logits = readouts.center.logits(centered[rows])\n    return claude, _score_frame([ids[row] for row in rows], np.exp(_log_softmax(logits)), LABELS)\n\n\ndef _score_frame(ids: list[str], values: Array, labels: tuple[str, ...]) -> pd.DataFrame:\n    _require_finite(values, (len(ids), len(labels)), "auxiliary scores")\n    frame = pd.DataFrame(values, columns=labels)\n    frame.insert(0, "id", pd.Series(ids, dtype=object))\n    return frame\n\n\ndef _log_softmax(logits: Array) -> Array:\n    shifted = logits - logits.max(axis=1, keepdims=True)\n    return shifted - np.log(np.exp(shifted).sum(axis=1, keepdims=True))\n\n\ndef fuse_claude_branch(\n    plain: pd.DataFrame, centered: pd.DataFrame, gate: pd.DataFrame\n) -> pd.DataFrame:\n    ids = plain["id"].tolist()\n    if (\n        tuple(plain.columns) != ("id", *LABELS)\n        or tuple(centered.columns) != ("id", *LABELS)\n        or tuple(gate.columns) != ("id", *CLAUDE_LABELS)\n        or any(frame["id"].duplicated().any() for frame in (plain, centered, gate))\n        or set(gate["id"]) != set(ids)\n        or not set(centered["id"]).issubset(ids)\n    ):\n        raise ContractError("fusion outputs do not align by ID and class")\n    logits = 0.177 * np.log(np.clip(plain.iloc[:, 1:].to_numpy(dtype=np.float64), 1e-9, 1.0))\n    positions = {record: row for row, record in enumerate(ids)}\n    rows = [positions[record] for record in centered["id"]]\n    logits[rows] += 0.15 * np.log(\n        np.clip(centered.iloc[:, 1:].to_numpy(dtype=np.float64), 1e-9, 1.0)\n    )\n    columns = [LABELS.index(label) for label in CLAUDE_LABELS]\n    base = logits[:, columns]\n    conditional = _log_softmax(base)\n    ranked = np.sort(np.exp(conditional), axis=1)\n    strength = 0.45 * np.clip(1.0 - (ranked[:, -1] - ranked[:, -2]) / 0.45, 0, 1)\n    guide = _log_softmax(gate.set_index("id").loc[ids].to_numpy(dtype=np.float64))\n    mixed = _log_softmax(conditional * (1 - strength[:, None]) + guide * strength[:, None])\n    logits[:, columns] = (base[:, :1] - conditional[:, :1]) + mixed\n    return _score_frame(ids, np.exp(_log_softmax(logits)), LABELS)\n\n\ndef encode_passages(\n    tokenizer: Callable[..., dict[str, list[list[int]]]], texts: list[str], eos_id: int\n) -> list[list[int]]:\n    encoded = tokenizer(texts, add_special_tokens=False, truncation=True, max_length=TOKEN_LIMIT)[\n        "input_ids"\n    ]\n    if any(not ids for ids in encoded):\n        raise ContractError("a passage produced no tokens before EOS and cannot be pooled")\n    return [ids + [eos_id] for ids in encoded]\n\n\ndef verify_anchors(observed: Array, saved: Array) -> float:\n    shape = (3, FEATURE_SIZE)\n    _require_finite(observed, shape, "observed anchor features")\n    _require_finite(saved, shape, "saved anchor features")\n    error = float(np.abs(observed - saved).mean() / (np.abs(saved).mean() + 1e-9))\n    if error >= 0.02:\n        raise ContractError(f"checkpoint anchor parity failed (relative error {error:.6g})")\n    return error\n\n\ndef branch_outputs(test: pd.DataFrame, logits: Array) -> tuple[pd.DataFrame, pd.DataFrame]:\n    test = validate_test(test)\n    _require_finite(logits, (len(test), len(LABELS)), "branch logits")\n    exp = np.exp(logits - logits.max(axis=1, keepdims=True))\n    scores = pd.DataFrame(exp / exp.sum(axis=1, keepdims=True), columns=list(LABELS))\n    scores.insert(0, "id", test["id"])\n    predictions = pd.DataFrame({"id": test["id"], "model": np.asarray(LABELS)[logits.argmax(1)]})\n    return scores, validate_submission(predictions, test)\n', 'src/ntl_repro/gpu.py': '"""Load the author\'s local checkpoint and reproduce the plain classifier readout."""\n\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport torch\nfrom peft import PeftModel, get_peft_model_state_dict\nfrom safetensors import safe_open\nfrom safetensors.torch import load_file\nfrom transformers import AutoConfig, AutoModel, AutoTokenizer\n\nfrom ntl_baseline.constants import ContractError\n\nfrom .branch import FEATURE_SIZE, HIDDEN_SIZE, POOL_LAYERS, Array, encode_passages\n\n\nclass CudaFeatures:\n    def __init__(self, base: Path, checkpoint: Path) -> None:\n        self.device = torch.device("cuda:0")\n        config = AutoConfig.from_pretrained(base, local_files_only=True, trust_remote_code=False)\n        keep = POOL_LAYERS[-1]\n        if (\n            config.model_type != "qwen3"\n            or config.hidden_size != HIDDEN_SIZE\n            or config.num_hidden_layers < keep\n            or getattr(config, "quantization_config", None) is not None\n        ):\n            raise ContractError("expected an unquantized Qwen3 trunk with 4096 hidden units")\n        config.num_hidden_layers = keep\n        config.use_cache = False\n        if getattr(config, "layer_types", None) is not None:\n            config.layer_types = config.layer_types[:keep]\n        trunk, info = AutoModel.from_pretrained(\n            base,\n            config=config,\n            dtype=torch.float16,\n            device_map={"": 0},\n            attn_implementation="sdpa",\n            use_safetensors=True,\n            local_files_only=True,\n            trust_remote_code=False,\n            output_loading_info=True,\n        )\n        if any(info.get(key) for key in ("missing_keys", "mismatched_keys", "error_msgs")):\n            raise ContractError("base checkpoint did not load all truncated trunk weights")\n        self.model = PeftModel.from_pretrained(\n            trunk, checkpoint / "adapter", local_files_only=True\n        ).eval()\n        with safe_open(\n            checkpoint / "adapter" / "adapter_model.safetensors", framework="pt"\n        ) as weights:\n            loaded_keys = get_peft_model_state_dict(self.model, save_embedding_layers=False)\n            if set(weights.keys()) != set(loaded_keys):\n                raise ContractError("adapter tensor keys differ from the loaded LoRA modules")\n\n        self.tokenizer = AutoTokenizer.from_pretrained(\n            checkpoint / "tokenizer", local_files_only=True, trust_remote_code=False\n        )\n        self.tokenizer.padding_side = "right"\n        if self.tokenizer.eos_token_id is None:\n            raise ContractError("checkpoint tokenizer is missing EOS")\n        if self.tokenizer.pad_token_id is None:\n            self.tokenizer.pad_token = self.tokenizer.eos_token\n\n        state = load_file(checkpoint / "heads.safetensors")\n        self.norms = torch.nn.ModuleList(torch.nn.LayerNorm(HIDDEN_SIZE) for _ in POOL_LAYERS)\n        self.norms.load_state_dict(\n            {\n                key.removeprefix("norms."): value\n                for key, value in state.items()\n                if key.startswith("norms.")\n            }\n        )\n        self.norms.to(device=self.device, dtype=torch.float32).eval()\n\n    @torch.no_grad()\n    def __call__(self, texts: list[str], batch_size: int) -> Array:\n        ids = encode_passages(self.tokenizer, texts, self.tokenizer.eos_token_id)\n        order = np.argsort([len(tokens) for tokens in ids], kind="stable")\n        features = np.empty((len(texts), FEATURE_SIZE), dtype=np.float32)\n        for start in range(0, len(order), batch_size):\n            rows = order[start : start + batch_size]\n            batch = self.tokenizer.pad(\n                {"input_ids": [ids[row] for row in rows]}, return_tensors="pt"\n            ).to(self.device)\n            features[rows] = self._pool(batch)\n        return features\n\n    def _pool(self, batch: Any) -> Array:\n        with torch.autocast("cuda", dtype=torch.float16):\n            hidden = self.model(**batch, output_hidden_states=True, use_cache=False).hidden_states\n        mask = batch["attention_mask"].unsqueeze(-1).float()\n        mask[:, 0] = 0\n        counts = mask.sum(dim=1)\n        pooled = [\n            norm((hidden[layer].float() * mask).sum(dim=1) / counts)\n            for layer, norm in zip(POOL_LAYERS, self.norms, strict=True)\n        ]\n        return torch.cat(pooled, dim=1).cpu().numpy()\n', 'src/ntl_repro/notebook.py': '"""Build an offline Notebook for the reproduced plain LoRA member."""\n\nimport hashlib\nfrom collections.abc import Callable\nfrom pathlib import Path\nfrom typing import Any\n\nimport pandas as pd\n\nfrom ntl_baseline.constants import ContractError\nfrom ntl_baseline.data import load_test, validate_submission\n\nfrom .branch import Array, branch_outputs\n\n\ndef write_plain_submission(\n    data_dir: Path, output: Path, score: Callable[[list[str]], Array]\n) -> int:\n    test = load_test(data_dir)\n    _, predictions = branch_outputs(test, score(test["text"].tolist()))\n    with output.open("xb") as stream:\n        predictions.to_parquet(stream, index=False)\n    saved = pd.read_parquet(output)\n    validate_submission(saved, test)\n    if not saved.equals(predictions):\n        raise ContractError("submission differs after Parquet readback")\n    return len(saved)\n\n\ndef notebook_document() -> dict[str, Any]:\n    from ntl_baseline.notebook import _cell\n\n    root = Path(__file__).resolve().parents[2]\n    files = (\n        "src/ntl_baseline/__init__.py",\n        "src/ntl_baseline/constants.py",\n        "src/ntl_baseline/data.py",\n        "src/ntl_repro/__init__.py",\n        "src/ntl_repro/branch.py",\n        "src/ntl_repro/gpu.py",\n        "src/ntl_repro/notebook.py",\n        "src/ntl_repro/run.py",\n        "configs/v31_single_gpu.toml",\n    )\n    sources = {name: (root / name).read_text(encoding="utf-8") for name in files}\n    hashes = {name: hashlib.sha256(value.encode()).hexdigest() for name, value in sources.items()}\n    bootstrap = """import hashlib\nimport json\nimport os\nimport sys\nimport tempfile\nfrom pathlib import Path\n\nos.environ.update(HF_HUB_OFFLINE=\'1\', TRANSFORMERS_OFFLINE=\'1\')\nif \'ntl_repro\' in sys.modules or \'ntl_baseline\' in sys.modules:\n    raise RuntimeError(\'Restart the kernel before running this Notebook.\')\n"""\n    bootstrap += f"SOURCES = {sources!r}\\nCHECKSUMS = {hashes!r}\\n"\n    bootstrap += """\nBUNDLE = tempfile.TemporaryDirectory(prefix=\'ntl-v31-plain-\')\nROOT = Path(BUNDLE.name)\nfor name, source in SOURCES.items():\n    if hashlib.sha256(source.encode()).hexdigest() != CHECKSUMS[name]:\n        raise RuntimeError(f\'Source checksum mismatch: {name}\')\n    path = ROOT / name\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(source, encoding=\'utf-8\')\nsys.path.insert(0, str(ROOT / \'src\'))\n"""\n    execute = """from unittest.mock import patch\n\nimport torch\n\nfrom ntl_repro.branch import load_readout\nfrom ntl_repro.gpu import CudaFeatures\nfrom ntl_repro.notebook import write_plain_submission\nfrom ntl_repro.run import check_assets, load_config\n\ndef locate(candidates, evidence, label):\n    matching = [path for path in candidates if (path / evidence).is_file()]\n    if len(matching) != 1:\n        raise RuntimeError(f\'Attach exactly the pinned {label} input; found {len(matching)}.\')\n    return matching[0]\n\nINPUT = Path(\'/kaggle/input\')\nDATA = locate([INPUT / \'competitions/name-that-llm\', INPUT / \'name-that-llm\'],\n              \'test.csv\', \'Name That LLM competition\')\nBASE = locate([INPUT / \'models/qwen-lm/qwen-3/transformers/8b-base/1\',\n               INPUT / \'models/qwen-lm/qwen-3/Transformers/8b-base/1\'],\n              \'model.safetensors.index.json\', \'Qwen3-8B Base v1\')\nCHECKPOINT = locate([\n    INPUT / \'datasets/zbigniewkirchner/ntl-qwen3-8b-lora-claude-head/final\',\n    INPUT / \'datasets/zbigniewkirchner/ntl-qwen3-8b-lora-claude-head/versions/1/final\',\n    INPUT / \'ntl-qwen3-8b-lora-claude-head/final\',\n], \'COMPLETE.json\', \'LoRA adapter v1\')\nif not torch.cuda.is_available():\n    raise RuntimeError(\'Select a GPU accelerator.\')\ntorch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False\nconfig = load_config(ROOT / \'configs/v31_single_gpu.toml\')\nasset_hashes = check_assets(BASE, CHECKPOINT, config)\nreadout = load_readout(CHECKPOINT / \'readouts.npz\')\nwith patch(\'peft.tuners.lora.torchao.is_torchao_available\', return_value=False):\n    extractor = CudaFeatures(BASE, CHECKPOINT)\n\ndef score(texts):\n    return readout.logits(extractor(texts, config.batch_size))\n\nrows = write_plain_submission(DATA, Path(\'/kaggle/working/submission.parquet\'), score)\nprint(json.dumps({\'rows\': rows, \'gpu\': torch.cuda.get_device_name(0),\n                  \'peak_cuda_reserved_bytes\': torch.cuda.max_memory_reserved(),\n                  \'asset_sha256\': asset_hashes}))\n"""\n    return {\n        "nbformat": 4,\n        "nbformat_minor": 5,\n        "metadata": {\n            "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},\n            "language_info": {"name": "python"},\n            "ntl_v31_source_sha256": hashes,\n        },\n        "cells": [\n            _cell(\n                "markdown",\n                """# Name That LLM — V31 plain LoRA member\n\nIndependent port of the plain `lora24_claude` member from\n[V31](https://www.kaggle.com/kernels/scriptcontent/372363760/download).\nAttach the competition, Qwen3-8B Base v1 and Zbigniew Kirchner\'s Apache 2.0\nLoRA dataset v1. Use a GPU with Internet disabled. The Notebook scores the\nmounted test rows directly and does not fit on competition training data.\n""",\n                "overview",\n            ),\n            _cell("code", bootstrap, "bundle-sources"),\n            _cell("code", execute, "plain-inference"),\n        ],\n    }\n', 'src/ntl_repro/run.py': '"""Offline server entry point. Every attempted inference writes to a new directory."""\n\nimport argparse\nimport json\nimport re\nimport sys\nimport tomllib\nfrom dataclasses import dataclass\nfrom importlib.metadata import version\nfrom pathlib import Path\nfrom time import perf_counter\n\nimport numpy as np\nimport pandas as pd\n\nfrom ntl_baseline.constants import LABELS, ContractError\nfrom ntl_baseline.data import file_sha256, load_test, read_csv_input, validate_train\n\nfrom .branch import (\n    auxiliary_outputs,\n    branch_outputs,\n    fuse_claude_branch,\n    load_auxiliary_readouts,\n    load_readout,\n    verify_anchors,\n)\n\nROOT = Path(__file__).resolve().parents[2]\n# Pinnable small files from Kaggle and named weight files from the same release.\nCHECKPOINT_FILES = (\n    "COMPLETE.json",\n    "adapter/adapter_config.json",\n    "adapter/adapter_model.safetensors",\n    "heads.safetensors",\n    "readouts.npz",\n    "tokenizer/tokenizer.json",\n    "tokenizer/tokenizer_config.json",\n    "anchor_features.npz",\n)\n\n\n@dataclass(frozen=True)\nclass Config:\n    batch_size: int\n    checkpoint_hashes: dict[str, str]\n    base_hashes: dict[str, str]\n    base_source: str\n\n\ndef load_config(path: Path) -> Config:\n    with path.open("rb") as stream:\n        data = tomllib.load(stream)\n    if (\n        data.get("script_version") != 372363760\n        or data.get("branch") != "lora24_claude"\n        or data.get("checkpoint") != "zbigniewkirchner/ntl-qwen3-8b-lora-claude-head/1"\n        or data.get("recommended_base") != "qwen-lm/qwen-3/Transformers/8b-base/1"\n    ):\n        raise ContractError("expected the pinned V31 lora24_claude configuration")\n    batch_size = data.get("batch_size")\n    if type(batch_size) is not int or not 1 <= batch_size <= 8:\n        raise ContractError("batch_size must be an integer from 1 to 8")\n    checkpoint_hashes = _parse_hashes(\n        data.get("sha256"), set(CHECKPOINT_FILES) - {"adapter/adapter_model.safetensors"}\n    )\n    base_hashes = _parse_hashes(\n        data.get("base_sha256"), {"config.json", "model.safetensors.index.json"}\n    )\n    return Config(batch_size, checkpoint_hashes, base_hashes, data["recommended_base"])\n\n\ndef _parse_hashes(value: object, names: set[str]) -> dict[str, str]:\n    if not isinstance(value, dict) or set(value) != names:\n        raise ContractError("configuration must pin all required control file hashes")\n    if not all(\n        isinstance(digest, str) and re.fullmatch(r"[0-9a-f]{64}", digest)\n        for digest in value.values()\n    ):\n        raise ContractError("configuration contains an invalid SHA-256")\n    return value\n\n\ndef _require_files(directory: Path, names: tuple[str, ...]) -> None:\n    for name in names:\n        if not (directory / name).is_file():\n            raise ContractError(f"required asset is missing: {name}")\n\n\ndef check_assets(base: Path, checkpoint: Path, config: Config) -> dict[str, dict[str, str]]:\n    _require_files(checkpoint, CHECKPOINT_FILES)\n    _require_files(base, tuple(config.base_hashes))\n    for directory, expected in (\n        (base, config.base_hashes),\n        (checkpoint, config.checkpoint_hashes),\n    ):\n        for name, digest in expected.items():\n            if file_sha256(directory / name) != digest:\n                raise ContractError(f"pinned SHA-256 mismatch: {name}")\n    if (base / "model.safetensors").exists():\n        raise ContractError("base directory must use the pinned sharded safetensors layout")\n    index = json.loads((base / "model.safetensors.index.json").read_text())\n    shards = tuple(sorted(set(index["weight_map"].values())))\n    if not shards or any(\n        not re.fullmatch(r"model-\\d{5}-of-\\d{5}\\.safetensors", name) for name in shards\n    ):\n        raise ContractError("unexpected base checkpoint shard names")\n    _require_files(base, shards)\n    adapter = checkpoint / "adapter" / "adapter_model.safetensors"\n    if adapter.stat().st_size != 116434760:\n        raise ContractError("adapter weight size differs from Kaggle v1 metadata")\n    if (checkpoint / "adapter" / "adapter_model.bin").exists():\n        raise ContractError("remove the unrecognized adapter .bin file from the asset directory")\n    return {\n        "base": {name: file_sha256(base / name) for name in (*config.base_hashes, *shards)},\n        "checkpoint": {name: file_sha256(checkpoint / name) for name in CHECKPOINT_FILES},\n    }\n\n\ndef labeled_preview(train: pd.DataFrame, count: int) -> pd.DataFrame:\n    prompts = train["prompt_id"]\n    available = prompts[~prompts.isin(prompts.head(3))].drop_duplicates()\n    if len(available) < count:\n        raise ContractError("not enough complete prompts outside the anchor prompts")\n    return train.loc[prompts.isin(available.head(count))].reset_index(drop=True)\n\n\ndef _run(args: argparse.Namespace) -> None:\n    config = load_config(args.config)\n    train = (\n        validate_train(read_csv_input(args.anchor_data_dir, "train.csv"))\n        if args.anchor_data_dir is not None\n        else None\n    )\n    preview = None\n    if args.preview_prompts is not None:\n        if train is None:\n            raise ContractError("labeled preview requires the original training CSV")\n        preview = labeled_preview(train, args.preview_prompts)\n        test = preview[["id", "text"]]\n    elif args.synthetic:\n        test = pd.DataFrame(\n            {\n                "id": ["synthetic-002", "synthetic-001", "synthetic-003"],\n                "text": [\n                    "A synthetic sentence for the first CUDA check.",\n                    "Second sample.",\n                    "Synthetic prose. " * 500,\n                ],\n            }\n        )\n    else:\n        test = load_test(args.data_dir)\n    if args.limit is not None:\n        test = test.head(args.limit)\n    run_dir = args.run_dir.resolve()\n    if not run_dir.is_relative_to(ROOT / "artifacts") or run_dir == ROOT / "artifacts":\n        raise ContractError("run-dir must be a new directory under this checkout\'s artifacts/")\n    if run_dir.exists():\n        raise ContractError("run-dir already exists")\n    hashes = check_assets(args.base_dir, args.checkpoint_dir, config)\n    readout_path = args.checkpoint_dir / "readouts.npz"\n    readout = load_readout(readout_path)\n    auxiliary = (\n        load_auxiliary_readouts(readout_path) if args.auxiliary or preview is not None else None\n    )\n\n    import torch\n\n    from .gpu import CudaFeatures\n\n    if not torch.cuda.is_available() or torch.cuda.device_count() != 1:\n        raise ContractError("expose exactly one CUDA GPU for inference")\n\n    anchor_texts = train["text"].head(3).tolist() if train is not None else None\n\n    run_dir.mkdir(parents=True, exist_ok=False)\n    torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False\n    torch.cuda.synchronize()\n    torch.cuda.reset_peak_memory_stats()\n    started = perf_counter()\n    extractor = CudaFeatures(args.base_dir, args.checkpoint_dir)\n    torch.cuda.synchronize()\n    load_seconds = perf_counter() - started\n\n    anchor = None\n    if anchor_texts is not None:\n        with np.load(args.checkpoint_dir / "anchor_features.npz", allow_pickle=False) as saved:\n            anchor = verify_anchors(extractor(anchor_texts, 3), saved["features"])\n\n    torch.cuda.synchronize()\n    prediction_started = perf_counter()\n    features = extractor(test["text"].tolist(), config.batch_size)\n    scores, predictions = branch_outputs(test, readout.logits(features))\n    outputs = {"probabilities.parquet": scores, "predictions.parquet": predictions}\n    preview_metrics = None\n    grouping = None\n    if auxiliary is not None:\n        groups = ["pair", "pair", "single"][: len(test)] if args.synthetic else None\n        if preview is not None:\n            groups = preview["prompt_id"].tolist()\n            grouping = "official_train_prompt_ids"\n        elif args.synthetic:\n            grouping = "synthetic_fixture"\n        gate, twin = auxiliary_outputs(test["id"].tolist(), features, auxiliary, groups)\n        outputs["claude_gate_logprobs.parquet"] = gate\n        if groups is not None:\n            outputs["centered_probabilities.parquet"] = twin\n        if preview is not None:\n            fused = fuse_claude_branch(scores, twin, gate)\n            fused_pred = np.asarray(LABELS)[fused.iloc[:, 1:].to_numpy().argmax(axis=1)]\n            truth = preview["model"].to_numpy()\n            preview_metrics = {\n                "prompts": args.preview_prompts,\n                "classes_present": int(preview["model"].nunique()),\n                "plain_balanced_accuracy": float(\n                    pd.Series(predictions["model"].to_numpy() == truth).groupby(truth).mean().mean()\n                ),\n                "fused_balanced_accuracy": float(\n                    pd.Series(fused_pred == truth).groupby(truth).mean().mean()\n                ),\n            }\n            outputs["fused_probabilities.parquet"] = fused\n    torch.cuda.synchronize()\n    finished_prediction = perf_counter()\n    prediction_seconds = finished_prediction - prediction_started\n    for name, frame in outputs.items():\n        frame.to_parquet(run_dir / name, index=False)\n        if not pd.read_parquet(run_dir / name).equals(frame):\n            raise ContractError(f"Parquet round trip differs: {name}")\n\n    report = {\n        "script_version": 372363760,\n        "branch": "lora24_claude",\n        "member": "plain",\n        "base_source": config.base_source,\n        "checkpoint_source": "zbigniewkirchner/ntl-qwen3-8b-lora-claude-head/1",\n        "asset_sha256": hashes,\n        "config_sha256": file_sha256(args.config),\n        "outputs_sha256": {name: file_sha256(run_dir / name) for name in outputs},\n        "rows": len(test),\n        "synthetic": args.synthetic,\n        "classes": list(scores.columns[1:]),\n        "batch_size": config.batch_size,\n        "anchor_relative_error": anchor,\n        "auxiliary_rows": {\n            name: len(frame)\n            for name, frame in outputs.items()\n            if name not in ("probabilities.parquet", "predictions.parquet")\n        },\n        "centering_groups": grouping,\n        "labeled_preview": preview_metrics,\n        "gpu": torch.cuda.get_device_name(),\n        "cuda_build": torch.version.cuda,\n        "python": sys.version.split()[0],\n        "packages": {\n            package: version(package)\n            for package in (\n                "torch",\n                "transformers",\n                "accelerate",\n                "peft",\n                "safetensors",\n                "tokenizers",\n                "numpy",\n                "pandas",\n                "pyarrow",\n            )\n        },\n        "model_load_seconds": load_seconds,\n        "prediction_seconds": prediction_seconds,\n        "load_through_prediction_seconds": finished_prediction - started,\n        "peak_cuda_allocated_bytes": torch.cuda.max_memory_allocated(),\n        "peak_cuda_reserved_bytes": torch.cuda.max_memory_reserved(),\n    }\n    marker = run_dir / ".completed.tmp"\n    marker.write_text(json.dumps(report, indent=2) + "\\n", encoding="utf-8")\n    marker.rename(run_dir / "completed.json")\n    print(f"Saved {len(test)} rows of branch probabilities and predictions.")\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    data = parser.add_mutually_exclusive_group(required=True)\n    data.add_argument("--synthetic", action="store_true")\n    data.add_argument("--data-dir", type=Path)\n    data.add_argument("--preview-prompts", type=int)\n    parser.add_argument("--anchor-data-dir", type=Path)\n    parser.add_argument("--auxiliary", action="store_true", help="emit the published extra heads")\n    parser.add_argument("--base-dir", type=Path, required=True)\n    parser.add_argument("--checkpoint-dir", type=Path, required=True)\n    parser.add_argument("--config", type=Path, default=ROOT / "configs/v31_single_gpu.toml")\n    parser.add_argument("--run-dir", type=Path, required=True)\n    parser.add_argument("--limit", type=int)\n    args = parser.parse_args()\n    if args.limit is not None and args.limit < 1:\n        parser.error("--limit must be positive")\n    if args.preview_prompts is not None and (\n        not 1 <= args.preview_prompts <= 12 or args.limit is not None\n    ):\n        parser.error("--preview-prompts must be 1–12 and cannot be combined with --limit")\n    try:\n        _run(args)\n    except ContractError as exc:\n        parser.exit(1, f"V31 inference failed: {exc}\\n")\n    except ImportError as exc:\n        parser.exit(1, f"V31 GPU environment import failed ({type(exc).__name__}).\\n")\n    except (OSError, ValueError, KeyError, RuntimeError) as exc:\n        parser.exit(\n            1, f"V31 inference failed ({type(exc).__name__}). No completion marker written.\\n"\n        )\n\n\nif __name__ == "__main__":\n    main()\n', 'configs/v31_single_gpu.toml': '# Kaggle scriptVersionId 372363760. Plain member of the comp_stack ensemble.\nbranch = "lora24_claude"\nscript_version = 372363760\ncheckpoint = "zbigniewkirchner/ntl-qwen3-8b-lora-claude-head/1"\nrecommended_base = "qwen-lm/qwen-3/Transformers/8b-base/1"\nbatch_size = 1\n\n# SHA-256 from the individual public v1 files, verified without downloading LoRA weights.\n[sha256]\n"COMPLETE.json" = "4a9f1ee827240c221c2e83bc0686df644aa3d16c2cef08e0b5e1e71c463b2a54"\n"adapter/adapter_config.json" = "e37c54313ad1d0886f0f4562e82e7934cea51b9433f7465fca9df9f765fe6176"\n"heads.safetensors" = "79fcbbc4b313450b4580565d5addba4cd769e80f10d5d68a2068e694b50619aa"\n"readouts.npz" = "39d8154724339e7b0fcf307267c588852cc4840ade7c4e59d7077a5cd55ea45b"\n"tokenizer/tokenizer_config.json" = "2e3bd3cb5055f7a903a54e7062b2fd14d21b70f4174c3ffcd9fc81ec50a25631"\n"anchor_features.npz" = "556cfc1a0a64915e99a8338789badb675496b910546cb2d2fdc8a5a9df69c4df"\n"tokenizer/tokenizer.json" = "b3744d3112b8c3d04760859bfdbf9a7ce00d5539512b5fc5d5bfb3f76eec0d9d"\n\n[base_sha256]\n"config.json" = "3bd01d7ad7a2e203ecbbe84e24087a51c6d2a108ee4bcc42d0016bf49564983a"\n"model.safetensors.index.json" = "cd39d5e44c5aadfd1954b598883aea1c4a98822a2f7723d214ef2278b7fec2e4"\n'}
CHECKSUMS = {'src/ntl_baseline/__init__.py': '534e449b7b5d310d159db644162ba10f7f2c0e0ec33f616e4cb954be1d072639', 'src/ntl_baseline/constants.py': '9149e8db8973ffdca1cc67452ca105efc81c1ea4677ae8cbb9d7f497d6c661bc', 'src/ntl_baseline/data.py': '0d03ef59f14f60ce7f4a96d999292e6753bc1763455734c59fba204acbb66c7e', 'src/ntl_repro/__init__.py': 'fbce995e1a16d9c0fc279cd3adecf36b200b5d29b511b8ef6b1809ffd01528d7', 'src/ntl_repro/branch.py': '7b0caf463f2191d54a52790ce1b7d685e8d544470fdf0d2ef0977a0efb03f6ae', 'src/ntl_repro/gpu.py': 'ac3b354706a7bf003028426325d794db26f38e2096190208999d833b0c97de8b', 'src/ntl_repro/notebook.py': '70cc6910cfb547ee876c741e05e183f00beb162440e47c56ee85a0a5af9af7f8', 'src/ntl_repro/run.py': '17e2d252c942457faa6d2cb802ee594f698a4831324b2ea1c12409911b86cc7a', 'configs/v31_single_gpu.toml': 'da4588bddb56e3b1dd9be513756680beb8a9d537bec226812bfd6d5f280c2189'}

BUNDLE = tempfile.TemporaryDirectory(prefix='ntl-v31-plain-')
ROOT = Path(BUNDLE.name)
for name, source in SOURCES.items():
    if hashlib.sha256(source.encode()).hexdigest() != CHECKSUMS[name]:
        raise RuntimeError(f'Source checksum mismatch: {name}')
    path = ROOT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source, encoding='utf-8')
sys.path.insert(0, str(ROOT / 'src'))


In [ ]:
from unittest.mock import patch

import torch

from ntl_repro.branch import load_readout
from ntl_repro.gpu import CudaFeatures
from ntl_repro.notebook import write_plain_submission
from ntl_repro.run import check_assets, load_config

def locate(candidates, evidence, label):
    matching = [path for path in candidates if (path / evidence).is_file()]
    if len(matching) != 1:
        raise RuntimeError(f'Attach exactly the pinned {label} input; found {len(matching)}.')
    return matching[0]

INPUT = Path('/kaggle/input')
DATA = locate([INPUT / 'competitions/name-that-llm', INPUT / 'name-that-llm'],
              'test.csv', 'Name That LLM competition')
BASE = locate([INPUT / 'models/qwen-lm/qwen-3/transformers/8b-base/1',
               INPUT / 'models/qwen-lm/qwen-3/Transformers/8b-base/1'],
              'model.safetensors.index.json', 'Qwen3-8B Base v1')
CHECKPOINT = locate([
    INPUT / 'datasets/zbigniewkirchner/ntl-qwen3-8b-lora-claude-head/final',
    INPUT / 'datasets/zbigniewkirchner/ntl-qwen3-8b-lora-claude-head/versions/1/final',
    INPUT / 'ntl-qwen3-8b-lora-claude-head/final',
], 'COMPLETE.json', 'LoRA adapter v1')
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU accelerator.')
torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False
config = load_config(ROOT / 'configs/v31_single_gpu.toml')
asset_hashes = check_assets(BASE, CHECKPOINT, config)
readout = load_readout(CHECKPOINT / 'readouts.npz')
with patch('peft.tuners.lora.torchao.is_torchao_available', return_value=False):
    extractor = CudaFeatures(BASE, CHECKPOINT)

def score(texts):
    return readout.logits(extractor(texts, config.batch_size))

rows = write_plain_submission(DATA, Path('/kaggle/working/submission.parquet'), score)
print(json.dumps({'rows': rows, 'gpu': torch.cuda.get_device_name(0),
                  'peak_cuda_reserved_bytes': torch.cuda.max_memory_reserved(),
                  'asset_sha256': asset_hashes}))
